# 01 - EDA: SROIE Receipt OCR + Key Information Extraction

**Project**: 12_sroie_invoice  
**Author**: Sandeep Grover, Independent Research  
**Goal**: explore the ICDAR 2019 SROIE dataset (1000 receipts) before any modeling. Inspect image dimensions, OCR-box coverage, ground-truth field statistics, and class balance for token-level BIO tagging.

Initial implementation - cells are code-only and have not been executed.

## 1. Setup and data loading skeleton

Expected structure after `kaggle datasets download -d urbikn/sroie-datasetv2`:

```
data/SROIE2019/{train,test}/{img,box,entities}/
```

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

DATA_ROOT = Path('../data/SROIE2019')
TRAIN_IMG = DATA_ROOT / 'train' / 'img'
TRAIN_BOX = DATA_ROOT / 'train' / 'box'
TRAIN_ENT = DATA_ROOT / 'train' / 'entities'
TEST_IMG  = DATA_ROOT / 'test'  / 'img'
TEST_ENT  = DATA_ROOT / 'test'  / 'entities'

print('train images :', len(list(TRAIN_IMG.glob('*.jpg'))) if TRAIN_IMG.exists() else 'NOT DOWNLOADED')
print('test  images :', len(list(TEST_IMG.glob('*.jpg')))  if TEST_IMG.exists()  else 'NOT DOWNLOADED')

## 2. Image dimension distribution

Receipts are typically narrow and tall (thermal-printer aspect ratios). Width/height percentiles drive the resize strategy for both Tesseract and LayoutLMv3 (which expects 224x224 patches over a 1000x1000 normalized canvas).

In [ ]:
rows = []
for p in sorted(TRAIN_IMG.glob('*.jpg')):
    with Image.open(p) as im:
        rows.append({'file': p.name, 'width': im.width, 'height': im.height})
df_img = pd.DataFrame(rows)
df_img.describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
df_img['width'].hist(ax=ax[0], bins=40)
ax[0].set_title('Receipt width (px)')
df_img['height'].hist(ax=ax[1], bins=40)
ax[1].set_title('Receipt height (px)')
plt.tight_layout(); plt.show()

## 3. Ground-truth field coverage

Each receipt has a JSON in `entities/` with the four target keys. We check missingness and string-length distributions per field. Missing fields are rare in SROIE but do exist (typically `address` on small thermal receipts where the merchant address was cropped).

In [ ]:
FIELDS = ['company', 'date', 'address', 'total']
ent_rows = []
for p in sorted(TRAIN_ENT.glob('*.txt')):
    try:
        ent = json.loads(p.read_text(encoding='utf-8'))
    except Exception:
        continue
    ent_rows.append({'file': p.stem, **{f: ent.get(f, '') for f in FIELDS}})
df_ent = pd.DataFrame(ent_rows)
df_ent.head()

In [ ]:
missing = (df_ent[FIELDS] == '').sum()
print('Missing per field (train):')
print(missing)
df_ent[FIELDS].applymap(len).describe()

## 4. Word-box statistics

Each `box/*.txt` file is a list of comma-separated rows of the form `x1,y1,x2,y2,x3,y3,x4,y4,text`. Average words per receipt and the per-receipt token count drive the LayoutLMv3 max-sequence-length decision (default 512; SROIE rarely exceeds it but a long supermarket receipt can).

In [ ]:
def parse_box_file(path: Path):
    out = []
    for line in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        parts = line.split(',', 8)
        if len(parts) < 9:
            continue
        coords = list(map(int, parts[:8]))
        text = parts[8].strip()
        out.append((coords, text))
    return out

lengths = []
for p in sorted(TRAIN_BOX.glob('*.txt')):
    lengths.append(len(parse_box_file(p)))
pd.Series(lengths, name='words_per_receipt').describe()

## 5. Date and total format diversity

SROIE receipts come from Singapore/Malaysia merchants - dates appear as `DD/MM/YYYY`, `DD-MM-YY`, `D MMM YYYY`, and a handful of free-form strings. Totals appear with `RM`, `MYR`, no symbol, and occasionally with thousands separators. The baseline regex needs to cover all of these; the advanced model learns them implicitly from the BIO labels.

In [ ]:
import collections
DATE_PATTERNS = [
    (r'^\d{1,2}[/-]\d{1,2}[/-]\d{2,4}$', 'numeric_slash_dash'),
    (r'^\d{1,2}\s+[A-Za-z]{3,}\s+\d{2,4}$', 'd_mon_y'),
    (r'^\d{4}[/-]\d{1,2}[/-]\d{1,2}$', 'iso_like'),
]
def classify_date(s):
    for pat, name in DATE_PATTERNS:
        if re.match(pat, s.strip()):
            return name
    return 'other'

if not df_ent.empty:
    print(collections.Counter(df_ent['date'].apply(classify_date)))

## 6. BIO label distribution (preview)

For LayoutLMv3 token classification we tag each OCR word with one of nine labels: `O`, `B-COMPANY`, `I-COMPANY`, `B-DATE`, `I-DATE`, `B-ADDRESS`, `I-ADDRESS`, `B-TOTAL`, `I-TOTAL`. Class imbalance is severe - O typically holds >85% of tokens. This drives the choice of weighted loss or focal loss in the advanced script.

In [ ]:
# Placeholder - actual BIO assignment lives in src/model_advanced.py
# Here we sketch what the per-class count will look like.
BIO_LABELS = ['O',
              'B-COMPANY','I-COMPANY',
              'B-DATE','I-DATE',
              'B-ADDRESS','I-ADDRESS',
              'B-TOTAL','I-TOTAL']
print('Label space size:', len(BIO_LABELS))

## 7. Open questions to resolve before modeling

- Should `address` be evaluated as a single normalized line or with internal newlines preserved? SROIE Task 3 expects normalized.
- Train/val split: SROIE provides only train/test. We carve a 10% validation set from train (seed=42) for early stopping during LayoutLMv3 fine-tune.
- Tesseract language pack: `eng` is sufficient for SROIE (English-language merchants). For DACH transfer the same pipeline switches to `deu+eng`.
- Image preprocessing: deskew + adaptive threshold (OpenCV) before Tesseract recovers ~3-5 F1 points on the baseline; the advanced model is more robust because it sees raw pixels via the visual backbone.